# ema-second-moment composite — cx21: v EMA then bias-correct: v_hat = v / (1 - beta2**t)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `ema-second-moment`, `bias-correction-divide`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "ema-second-moment"
DD_ATOM_IDS = ["ema-second-moment", "bias-correction-divide"]
DD_SUBTOPICS = ["Optimizer: Adam EMA second moment", "Optimizer: Adam bias-correction divide"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The second-moment EMA recurrence is structurally identical to the first-moment one, but EMAs the SQUARED gradient and uses a separate decay `beta2` (typically 0.999, much slower than `beta1 = 0.9`):
```
v = beta2 * v + (1 - beta2) * g.pow(2)
```
And the bias correction has the same shape, but the divisor uses `beta2`:
```
v_hat = v / (1 - beta2 ** t)
```
**Why a SEPARATE beta2.** `v` accumulates `g**2`, which is much noisier and skewed than `g` itself. A slower EMA (larger beta2) smooths it harder so the per-coordinate scale in the Adam update doesn't whip around batch-to-batch.

**The classic bug.** Applying the WRONG bias-correction divisor (using `beta1` instead of `beta2`). Since `beta1 != beta2`, the resulting `v_hat` is wrong by the ratio `(1 - beta2**t) / (1 - beta1**t)` — a multiplicative scale error that gets squared into the denominator. Test cases here pin this down.

**v_hat is always non-negative.** `v = EMA(g**2)` starts at 0 and adds only non-negative terms, so `v >= 0` elementwise. The divisor `1 - beta2**t` is positive for `t >= 1`, so `v_hat >= 0` too. Downstream code can safely take `sqrt(v_hat)`.

### Composite Exercise — v EMA then bias-correct: v_hat = v / (1 - beta2**t)

**Atoms exercised together**: `ema-second-moment`, `bias-correction-divide`

Implement `cx21_ema_v_then_bias_correct(v, g, beta2, t_step)`.

Inputs:
- `v`: current second-moment buffer (Tensor of any shape).
- `g`: current gradient (Tensor, same shape as `v`).
- `beta2`: float decay (typically 0.999).
- `t_step`: int >= 1.

Returns `(v_new, v_hat)`:
- `v_new = beta2 * v + (1 - beta2) * g.pow(2)` (atom: ema-second-moment).
- `v_hat = v_new / (1 - beta2 ** t_step)` (atom: bias-correction-divide).

Do not mutate `v` in place.

Tests verify:
- Step 1 with `v = 0`: `v_new = (1 - beta2) * g**2`, `v_hat = g**2` (closed form).
- `v_new` (and `v_hat`) is non-negative elementwise even when `g` is negative.
- Many constant-g steps: `v_hat == g**2` at every step.
- Cross-check vs `torch.optim.Adam`'s `exp_avg_sq` + the `1 - beta2**t` divisor.
- Sabotage: using `beta1=0.9` as the divisor (the classic 'used wrong beta' bug) FAILS by a known factor — the test confirms you used `beta2`.

In [ ]:
def cx21_ema_v_then_bias_correct(v, g, beta2, t_step):
    # Atom A (ema-second-moment): EMA on g**2 with decay beta2.
    v_new = beta2 * v + (1.0 - beta2) * g.pow(2)
    # Atom B (bias-correction-divide): use beta2 (NOT beta1) in the divisor.
    v_hat = v_new / (1.0 - beta2 ** t_step)
    return v_new, v_hat


<details><summary>Show solution — cx21</summary>

```python
def cx21_ema_v_then_bias_correct(v, g, beta2, t_step):
    # Atom A (ema-second-moment): EMA on g**2 with decay beta2.
    v_new = beta2 * v + (1.0 - beta2) * g.pow(2)
    # Atom B (bias-correction-divide): use beta2 (NOT beta1) in the divisor.
    v_hat = v_new / (1.0 - beta2 ** t_step)
    return v_new, v_hat
```

**Mirror of cx20, with two critical swaps:** `beta1 -> beta2`, `g -> g.pow(2)`. Same structural composition — the bias correction always uses the SAME beta as the EMA it corrects. Using `beta1` as the divisor for `v` is the textbook Adam-from-scratch bug.

**Why `g.pow(2)` not `g * g`.** Both produce the same value. `pow(2)` is fused in PyTorch's elementwise CUDA kernel and skips one tensor allocation. For Adam's inner loop this matters at large model scale.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx21'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx21',
        'subtopics': ["Optimizer: Adam EMA second moment", "Optimizer: Adam bias-correction divide"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()